In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

repo_root = _find_repo_root(Path.cwd())
workspace_root = repo_root.parent
candidate_src_dirs = [
    repo_root / "src",
    workspace_root / "abstractgraph" / "src",
    workspace_root / "abstractgraph-ml" / "src",
    workspace_root / "abstractgraph-generative" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [ ]:
from abstractgraph.to_graph.nlp_dependency import sentence_dependency_graph, display_dependency

sentences = [ "The swift red fox leaps over the sleepy hound.", 
             "The nimble gray cat bounds over the lazy beagle.", 
             "The spry white rabbit hops over the drowsy tortoise.", 
             "The agile black goat vaults over the sluggish sheep.", 
             "The speedy spotted cheetah springs over the tired zebra." ]

sentences = ['the cat chases the mouse.', 'the dog chases the cat.']

graphs = [sentence_dependency_graph(sentence) for sentence in sentences]
_ = display_dependency(graphs, n_graphs_per_line=2, anchor_offset=0.5,draw_source_stem=False)
graphs = [graph.to_undirected() for graph in graphs]

In [ ]:
from abstractgraph.display import display_graphs as abstract_display_graphs

def display_graphs(graphs):
    return abstract_display_graphs(graphs, node_labels=True, edge_labels=True, size=(6,6), node_label_font_size=7, edge_label_font_size=5, display_nodes=False)
display_graphs(graphs)

In [ ]:
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator

df = compose(neighborhood(radius=2), unlabel())
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
df = add(neighborhood(radius=1), cycle())
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
feasibility_estimators = [fe1, fe2]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

nbits=14
#decomposition_function = add(cycle(), neighborhood(radius=(0,2)))
#decomposition_function = add(node(), edge(), cycle())
#decomposition_function = add(node(), compose_product(binary_combination(distance=3), neighborhood(), node()))
#decomposition_function = add(node(), edge(), cycle(), neighborhood(radius=(1,2)))
decomposition_function = node()

graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=add(cycle(), neighborhood(radius=(0,2))),
    return_dense=True,
)
from abstractgraph_generative.autoregressive import AutoregressiveGraphGenerator
generator = AutoregressiveGraphGenerator(
    feasibility_estimator, 
    nbits, 
    decomposition_function, 
    cut_radius=None,
    max_cut_size=4, 
    max_virtual_cut_evaluations=30*30*30, #During generation, it samples one cut size at a time and evaluates up to 20*20*20=8000 virtual cut node sets per step.
    cut_context_radius=None, 
    context_vectorizer=graph_transformer,
    context_temperature=0, # Context selection is deterministic: context_temperature=0 + context_top_k=10 means it ranks all evaluated cut candidates by outer context cosine similarity 
    context_top_k=1, # and tries the top 10 in order (skipping any that fail)
    n_virtual_candidates=100, # each step proposes up to 100 rewrite candidates before selecting the next graph.
    use_context_embedding=True,
    min_nodes_for_pruning=4, 
    similarity_vectorizer=graph_transformer,
    similarity_k_neighbors=3,
    use_similarity_selection=True, # if True we select the our of the 100 the one most similar to the closest similarity_k_neighbors=3 generator graphs, otherwise we select the one with the most edges
    similarity_temperature=0,
    similarity_top_k=1,
    n_pruning_iterations=100, 
    verbose=True,
    display_function=display_graphs,
    display_live=True,
    n_jobs=-1)

from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph.operators import compose, neighborhood, cycle, add, unlabel
df = add(neighborhood(radius=(1,4)), cycle())
vec = AbstractGraphTransformer(nbits=nbits, decomposition_function=df, return_dense=True)
from abstractgraph_generative.repair import RepairGenerator
repair_generator = RepairGenerator(
    decomposition_function=df, 
    nbits=nbits, 
    graph_transformer=vec, 
    n_samples=64, 
    cut_radius=None, 
    cut_scope="outer", 
    single_replacement=True)

In [ ]:
import random
import time

def generate_graphs(graphs, generator, repair_generator, generator_size=1, n_samples=1, n_rounds=2, verbose=True):
    generator.store(graphs)
    generator_graphs = generator.get_generators(size=generator_size)
    if verbose: 
        print('Generator graphs:')
        display_graphs(generator_graphs)
    t0 = time.perf_counter()
    generator.fit(generator_graphs)
    t1 = time.perf_counter()
    if verbose:
        print(f'Generator fit time: {t1 - t0:.2f}s')
    t0 = time.perf_counter()
    repair_generator.fit(generator.donors)
    t1 = time.perf_counter()
    if verbose:
        print(f'RepairGenerator fit time: {t1 - t0:.2f}s')
    samples = [generator.donors[random.randrange(len(generator.donors))] for it in range(n_samples)]
    if verbose: 
        print('Seed graphs:')
        display_graphs(samples)
    for _ in range(n_rounds):
        samples = generator.generate_from(samples)
        if verbose: 
            print('Generated graphs before repair:')
            display_graphs(samples)
        samples = repair_generator.repair(samples, k_neighbors=2, n_iterations=3)
        if verbose: 
            print('Generated graphs after repair:')
            display_graphs(samples)
    samples = generator.generate_from(samples)
    if verbose: 
        print('Final graphs:')
        display_graphs(samples)
    return samples

---

In [ ]:
samples = generate_graphs(graphs, generator, repair_generator, generator_size=2, n_samples=3, n_rounds=2, verbose=True)

---